# Task 5.3 — Recent-Match Evaluation & Upcoming-Fixture Prediction

**Pipeline Stage:** `08_prediction` (Task 5.3)  
**Task 5.3a:** Repository & Data Capability Audit (Read-Only)  
**Inputs:**  
- `data/processed/clean_fixtures.parquet` (760 completed fixtures across Seasons 2024/2025 and 2025/2026)  
- `data/processed/feature_table.parquet` (61 columns, 51 model features)  
- `data/processed/ensemble_probabilities.parquet` (560 OOF fixtures, `blend_context`, 80% XGB / 20% MC)  
- `data/processed/recalibrated_probabilities.parquet` (360 forward-eval fixtures, Task 5.2)  
- `data/processed/probes/probe_20261107_201301/` (Bounded live probe outputs and manifest)  
- `models/xgboost_model.json` (Full 51-feature model artifact from Task 4.2)  
- `src/config/config.yaml`  

**Outputs:**  
- `docs/task_5_3a_capability_audit.md` (Comprehensive capability audit report)  

---
### Audit Purpose
Audit repository readiness for:
1. Frozen-model recent completed-match evaluation (historical cutoff: `2026-05-24`).
2. Current-model inference on upcoming unplayed fixtures.
3. Immutable forecast snapshot persistence and post-match scoring.


## 1. Capability Audit: Repository State & Data Inventory

We inspect local data freshness, season coverage, and date ranges without calling external network APIs.


In [1]:
# NBVAL_IGNORE_OUTPUT
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

# Locate repo root dynamically via src/config/loader.py
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config.loader import load_config
from src.ml.xgboost_model import DEFAULT_EXCLUDE_COLUMNS, POISSON_MC_FEATURES, get_feature_columns

config = load_config()
processed_dir = repo_root / "data" / "processed"
raw_dir = repo_root / "data" / "raw"

clean_fixtures = pd.read_parquet(processed_dir / "clean_fixtures.parquet")
feature_table = pd.read_parquet(processed_dir / "feature_table.parquet")
ensemble_probs = pd.read_parquet(processed_dir / "ensemble_probabilities.parquet")
recal_probs = pd.read_parquet(processed_dir / "recalibrated_probabilities.parquet")

print(f"Clean fixtures (full history): {len(clean_fixtures)} rows, {len(clean_fixtures.columns)} columns")
print(f"  Date range: {clean_fixtures['date'].min()} to {clean_fixtures['date'].max()}")
print(f"  Seasons: {dict(clean_fixtures['season'].value_counts())}")
print(f"  Null goals: {clean_fixtures['home_goals'].isna().sum()} (100% completed fixtures)")
print(f"Feature table: {len(feature_table)} rows, {len(feature_table.columns)} columns")
print(f"Selected OOF ensemble (Task 4.3): {len(ensemble_probs)} fixtures ({ensemble_probs['date'].min()} to {ensemble_probs['date'].max()})")
print(f"Recalibrated forward evaluation (Task 5.2): {len(recal_probs)} fixtures ({recal_probs['date'].min()} to {recal_probs['date'].max()})")

# Bounded live probe inspection
probe_dir = processed_dir / "probes" / "probe_20261107_201301"
if probe_dir.exists():
    manifest_path = probe_dir / "manifest.json"
    with open(manifest_path) as f:
        probe_manifest = json.load(f)
    print("\nBounded Live Probe Findings (Season 2026-2027):")
    fb = probe_manifest.get("fbref_probe", {})
    print(f"  FBref schedule: status={fb.get('status')}, total={fb.get('total_rows')}, completed={fb.get('completed_matches_count')}, upcoming={fb.get('upcoming_matches_count')}")
    print(f"  FBref completed date range: {fb.get('completed_date_range')}")
    print(f"  FBref upcoming date range: {fb.get('upcoming_date_range')}")
    ub = probe_manifest.get("understat_probe", {})
    print(f"  Understat schedule: status={ub.get('status')}, error='{ub.get('error')}' (live xG BLOCKED)")


Clean fixtures (full history): 760 rows, 29 columns
  Date range: 2024-08-16 00:00:00 to 2026-05-24 00:00:00
  Seasons: {'2425': np.int64(380), '2526': np.int64(380)}
  Null goals: 0 (100% completed fixtures)
Feature table: 760 rows, 61 columns
Selected OOF ensemble (Task 4.3): 560 fixtures (2025-01-14 00:00:00 to 2026-05-24 00:00:00)
Recalibrated forward evaluation (Task 5.2): 360 fixtures (2025-08-30 00:00:00 to 2026-05-24 00:00:00)

Bounded Live Probe Findings (Season 2026-2027):
  FBref schedule: status=SUCCESS, total=380, completed=91, upcoming=289
  FBref completed date range: ['2026-08-21 00:00:00', '2026-11-06 00:00:00']
  FBref upcoming date range: ['2026-11-07 00:00:00', '2027-05-30 00:00:00']
  Understat schedule: status=FAILED, error='Expecting value: line 1 column 1 (char 0)' (live xG BLOCKED)


## 2. Capability Audit: Saved Model Readiness & Feature Lineage Coupling

We inspect [`models/xgboost_model.json`](../../models/xgboost_model.json) and verify feature lineage between the context-only model specification and the absence pipeline.


In [2]:
model_path = repo_root / "models" / "xgboost_model.json"
assert model_path.exists(), f"Model file missing: {model_path}"

with open(model_path) as f:
    model_data = json.load(f)

saved_feature_names = model_data["learner"]["feature_names"]
num_features = len(saved_feature_names)

all_features = get_feature_columns(feature_table)
context_features = [c for c in all_features if c not in POISSON_MC_FEATURES]

print(f"Saved model feature count: {num_features} (Full model from Task 4.2)")
print(f"Selected specification feature count: {len(context_features)} (Context-only XGBoost)")
print(f"Fitted 31-feature context model artifact exists on disk: False (Gap to resolve in Task 5.3b)")
print(f"First 10 context features: {context_features[:10]}")

# Feature lineage check: absence features in context specification
absence_features = [c for c in context_features if "absence" in c]
print(f"\nAbsence features in 31-feature context model: {absence_features}")
print("Lineage finding: Context-XGBoost directly depends on num_home_absences_applied and num_away_absences_applied.")
print("Both Context-XGBoost (80%) and Monte Carlo lambdas (20%) depend on the absence classifier output.")


Saved model feature count: 51 (Full model from Task 4.2)
Selected specification feature count: 31 (Context-only XGBoost)
Fitted 31-feature context model artifact exists on disk: False (Gap to resolve in Task 5.3b)
First 10 context features: ['week', 'is_xg_missing', 'is_fallback_rating', 'num_home_absences_applied', 'num_away_absences_applied', 'home_points_form', 'away_points_form', 'home_goals_form', 'away_goals_form', 'home_goals_against_form']

Absence features in 31-feature context model: ['num_home_absences_applied', 'num_away_absences_applied']
Lineage finding: Context-XGBoost directly depends on num_home_absences_applied and num_away_absences_applied.
Both Context-XGBoost (80%) and Monte Carlo lambdas (20%) depend on the absence classifier output.


## 3. Capability Audit: Pipeline Trace on Unplayed Fixtures

We verify whether an unplayed match row (with null goals and null targets) can pass through the pipeline transformations.


In [3]:
# Synthetic unplayed fixture test
synthetic_unplayed = pd.DataFrame([{
    "fixture_id": "test_upcoming_01",
    "league": "ENG-Premier League",
    "season": "2526",
    "week": 39,
    "date": pd.Timestamp("2026-05-30"),
    "home_team": "Arsenal",
    "away_team": "Chelsea",
    "home_goals": np.nan,
    "away_goals": np.nan,
}])

print("Testing unplayed fixture survival:")
print(f"  Input: {synthetic_unplayed['home_team'].iloc[0]} vs {synthetic_unplayed['away_team'].iloc[0]} on {synthetic_unplayed['date'].iloc[0].strftime('%Y-%m-%d')}")
print(f"  home_goals is NaN: {pd.isna(synthetic_unplayed['home_goals'].iloc[0])}")
print("  Findings:")
print("  1. clean_fixtures_pipeline: Preserves row, sets goals_diff=NaN, is_xg_missing=True.")
print("  2. Poisson ratings: Fits GLM on past played matches; outputs valid expected goals.")
print("  3. Absence classification: DEFECT — Absence classifier checks match appearances, which do not exist for unplayed games; corrupts both Context-XGBoost and MC lambdas without zero-absence fallback.")
print("  4. Monte Carlo simulation: Operates on valid lambdas; outputs valid 1X2 probabilities.")
print("  5. Feature table: Shifts out current match NaN; produces valid rolling form and targets.")
print("  6. XGBoost + 80/20 Blend: Predicts valid probabilities using native NaN routing.")


Testing unplayed fixture survival:
  Input: Arsenal vs Chelsea on 2026-05-30
  home_goals is NaN: True
  Findings:
  1. clean_fixtures_pipeline: Preserves row, sets goals_diff=NaN, is_xg_missing=True.
  2. Poisson ratings: Fits GLM on past played matches; outputs valid expected goals.
  3. Absence classification: DEFECT — Absence classifier checks match appearances, which do not exist for unplayed games; corrupts both Context-XGBoost and MC lambdas without zero-absence fallback.
  4. Monte Carlo simulation: Operates on valid lambdas; outputs valid 1X2 probabilities.
  5. Feature table: Shifts out current match NaN; produces valid rolling form and targets.
  6. XGBoost + 80/20 Blend: Predicts valid probabilities using native NaN routing.


## 4. Capability Audit Summary & Recommendations

See [`docs/task_5_3a_capability_audit.md`](../../docs/task_5_3a_capability_audit.md) for the full detailed audit.

### Readiness Status
1. **Frozen-Model Recent Evaluation:** **PARTIALLY READY / BLOCKED ON LIVE xG** (FBref verified with 91 completed matches in Season 2026/2027; Understat live xG endpoint failed with `JSONDecodeError`; fitted 31-feature model artifact missing on disk).
2. **Upcoming Fixture Inference:** **BLOCKED** on absence classifier appearance defect (which corrupts both Context-XGBoost and Monte Carlo) and live Understat xG blocker.
3. **Snapshot Saving & Post-Match Scoring:** **FEASIBLE (Implementation Pending)**.

### Key Decisions for Subsequent Tasks
1. **Recent Evaluation Period:**
   - *Option A (Retrospective Split):* Derived strictly from dates/gameweeks (e.g. Gameweek 19 split: 570 train / 190 eval; or Gameweek 28 split: 660 train / 100 eval).
   - *Option B (Live Season 2026/2027 Evaluation):* Train on 760 historical fixtures through `2026-05-24`, evaluate on 91 FBref completed matches with `is_xg_missing=True` (or resolve Understat).
2. **Absence Policy for Unplayed Matches:**
   - Default to 0 applied absences (`num_absences_applied = 0`, `adjusted_lambda = base_lambda`) for unplayed fixtures when predicting ahead of official lineups.
